# Paid to Disagree: single-model BTS-GRPO (B9, Phase 2)

Trains Qwen2.5-1.5B with GRPO where the peers in the reward are frozen copies of Qwen itself
(`configs/train_b9.yaml`, `population: single_model`, BTS terms only). Phase 2 passes if dev
flip-to-wrong drops without a dev accuracy drop of more than 3 points.

**Notebook settings (right panel):** Accelerator **GPU T4 x2** (training uses one), Internet **on**.

**Recommended:** run steps 1-4 interactively first (the pilot: 40 questions, 20 GRPO steps).
It checks that Unsloth / TRL / vLLM install and run on the T4 and measures seconds per step.
If it passes, use **Save Version -> Save & Run All (Commit)** for the full run (step 5);
the pilot runs again in the commit, it only takes a few minutes. Download `b9_runs.zip` from
the Output tab when it finishes.

In [ ]:
# ---- settings ----
REPO = "https://github.com/nithish-74/paid-to-disagree.git"
CONFIG = "configs/train_b9.yaml"
PILOT = dict(max_train_questions=40, max_steps=20, monitor_every=10, dev_n=20)
RUN_FULL = True            # False: stop after the pilot

## 1. Code and packages

In [ ]:
import os, subprocess, json, time, pathlib, shutil
os.chdir("/kaggle/working")
if not pathlib.Path("ptd").exists():
    subprocess.run(["git", "clone", "-q", REPO, "ptd"], check=True)
os.chdir("/kaggle/working/ptd")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=True)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# Unsloth first (it pins compatible torch / transformers / trl), then vLLM for fast generation.
!pip install -q unsloth
!pip install -q vllm trl
!pip install -q -e .
!python -c "import unsloth, trl, vllm, torch; print('unsloth', unsloth.__version__, '| trl', trl.__version__, '| vllm', vllm.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

## 2. Question data

In [ ]:
!python -m data.build_questions --config configs/data.yaml
!wc -l data/processed/truthfulqa_mc1.jsonl data/processed/arc_challenge.jsonl data/processed/mmlu_pro.jsonl

## 3. Helpers

Each command runs in its own process (frees GPU memory between cache and train) and logs to
`runs/<experiment>.<command>.log`; the last lines are printed every few minutes.

In [ ]:
import yaml


def run_step(command, config, every=120, tail=15):
    exp = yaml.safe_load(open(config))["experiment"]
    log_path = pathlib.Path(f"runs/{exp}.{command}.log")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    start = time.time()
    with open(log_path, "a") as log:
        p = subprocess.Popen(["python", "-m", "train.grpo_roundrobin", command, "--config", config],
                             stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        while p.poll() is None:
            time.sleep(every)
            lines = [l.rstrip() for l in open(log_path, errors="replace") if l.strip()]
            print(f"[{(time.time() - start) / 60:5.1f} min] {command}: {lines[-1][:200] if lines else '...'}", flush=True)
    lines = [l.rstrip() for l in open(log_path, errors="replace") if l.strip()]
    print("\n".join(lines[-tail:]))
    print(f"{command} exit code {p.returncode} after {(time.time() - start) / 60:.1f} min")
    assert p.returncode == 0, f"{command} failed: see {log_path}"
    return time.time() - start


def show_leg(config):
    cfg = yaml.safe_load(open(config))
    exp = pathlib.Path("runs") / cfg["experiment"]
    for leg_dir in sorted(exp.glob("round*_*")):
        print(f"--- {leg_dir}")
        for name in ("monitor.jsonl", "reward_terms.jsonl"):
            f = leg_dir / name
            if f.exists():
                rows = [json.loads(l) for l in open(f) if l.strip()]
                print(f"{name}: {len(rows)} lines")
                for r in rows[:2] + rows[-2:]:
                    print("  ", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()
                                 if k not in ("history",)})
    print(subprocess.run(["python", "-m", "train.grpo_roundrobin", "status", "--config", config],
                         capture_output=True, text=True).stdout)

## 4. Gate: pilot (40 questions, 20 steps)

Same code path as the full run, at toy size: cache Qwen's samples, then a short GRPO leg with
a dev check every 10 steps. Passes if both steps exit cleanly and `reward_terms.jsonl` shows
every reward term. Seconds per step from here sets how many steps the full run can afford.

In [ ]:
cfg = yaml.safe_load(open(CONFIG))
pilot = {**cfg, "experiment": cfg["experiment"] + "_pilot",
         "max_train_questions": PILOT["max_train_questions"]}
pilot["grpo"] = {**cfg["grpo"], "max_steps": PILOT["max_steps"]}
pilot["monitor"] = {**cfg["monitor"], "every": PILOT["monitor_every"], "dev_n": PILOT["dev_n"]}
yaml.safe_dump(pilot, open("configs/train_b9_pilot.yaml", "w"), sort_keys=False)

shutil.rmtree(f"runs/{pilot['experiment']}", ignore_errors=True)   # a re-run starts clean
run_step("cache", "configs/train_b9_pilot.yaml", every=60)
t_train = run_step("train", "configs/train_b9_pilot.yaml", every=60)
show_leg("configs/train_b9_pilot.yaml")
print(f"pilot train: {t_train / 60:.1f} min for {PILOT['max_steps']} steps incl. model load and "
      f"{PILOT['max_steps'] // PILOT['monitor_every'] + 1} dev checks")

## 5. Full run: cache Qwen, then one 600-step leg

`--resume` is not needed: `state.json` records finished steps. The commit limit is 12 hours;
if the pilot showed the full leg will not fit, lower `grpo.max_steps` in `configs/train_b9.yaml`
(or here, before running) and say so in the paper.

In [ ]:
if RUN_FULL:
    run_step("cache", CONFIG)
    run_step("train", CONFIG, every=600)
    show_leg(CONFIG)
else:
    print("RUN_FULL is False: pilot only")

## 6. Test set: untrained vs B9 Qwen (Phase 2 comparison)

Qwen alone under every pressure condition, without (`configs/eval_b9_base.yaml`) and with the
B9 LoRA (`configs/eval_b9.yaml`), on the same seeds, then metrics and seed-combined tables.
Qwen runs on vLLM, so this takes minutes per seed, not hours.

In [ ]:
EVAL_SEEDS = [0, 1, 2]


def run_eval_seeds(config, prefix):
    runs = []
    for seed in EVAL_SEEDS:
        run_id = f"{prefix}_seed{seed}"
        if not pathlib.Path(f"runs/{run_id}/questions.jsonl").exists():
            t = time.time()
            r = subprocess.run(["python", "-m", "eval.run_eval", "--config", config, "--seed", str(seed),
                                "--run-id", run_id, "--resume"], capture_output=True, text=True)
            print(f"{run_id}: exit {r.returncode} after {(time.time() - t) / 60:.1f} min")
            assert r.returncode == 0, (r.stdout + r.stderr)[-3000:]
        subprocess.run(["python", "-m", "eval.metrics", f"runs/{run_id}", "--n-boot", "1000"], check=True,
                       capture_output=True)
        runs.append(f"runs/{run_id}")
    subprocess.run(["python", "-m", "analysis.combine_seeds", *runs, "--out", f"runs/{prefix}_combined"], check=True)
    return f"runs/{prefix}_combined"


if RUN_FULL:
    base = run_eval_seeds("configs/eval_b9_base.yaml", "eval_b9_base")
    b9 = run_eval_seeds("configs/eval_b9.yaml", "eval_b9")

    import pandas as pd
    rows = []
    for label, d in (("untrained", base), ("B9", b9)):
        m = pd.read_csv(f"{d}/members.csv")
        f = pd.read_csv(f"{d}/flips.csv")
        for ds in sorted(m.dataset.unique()):
            row = {"model": label, "dataset": ds,
                   "acc_neutral": m[(m.dataset == ds) & (m.condition == "neutral")].accuracy_mean.mean()}
            for cond, col in (("user_wrong", "cave_rate_mean"), ("peer_wrong", "cave_rate_mean"),
                              ("user_wrong", "flip_to_wrong_mean"), ("are_you_sure", "flip_to_wrong_mean"),
                              ("user_right", "flip_to_right_mean")):
                row[f"{cond}:{col.replace('_mean', '')}"] = f[(f.dataset == ds) & (f.condition == cond)][col].mean()
            rows.append(row)
    table = pd.DataFrame(rows).set_index(["dataset", "model"]).sort_index()
    print("Mean over seeds (fractions). Phase 2 passes if flip-to-wrong drops and accuracy drops <= 0.03.")
    print(table.round(3).to_string())

## 7. Package the results

In [ ]:
out = shutil.make_archive("/kaggle/working/b9_runs", "zip", "runs")
print(out, f"{os.path.getsize(out) / 1e6:.1f} MB")